# Greek LoRA fine-tuning on Colab: more capacity (Q3)
Same as the main run (3 epochs, learning rate 5e-4, rank 32), but LoRA on **all linear layers** (q, k, v, out, fc1, fc2: 13.0M trainable parameters instead of 3.5M). One change only, so the difference to `small-lora-el` is due to capacity. Compare on dev loss and dev WER only; do not tune on the test or rated sentences.

Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.
Upload `colab_package.zip` (the rebuilt one, it contains the new `--lora-targets` flag). The result is `small-lora-el-all.zip`: unzip it into `models/small-lora-el-all/`.

In [ ]:
!nvidia-smi | head -15

## 1. Environment (same versions as the laptop)

In [ ]:
# Colab ships torchao 0.10, which this peft version rejects; we do not use it
!pip -q uninstall -y torchao
!pip -q install transformers==5.22.1 peft==0.21.2 ctranslate2==4.8.2 whisper-normalizer jiwer soundfile huggingface_hub tabulate pandas

## 2. Upload `colab_package.zip` and unpack it

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick colab_package.zip
!mkdir -p /content/project && cd /content/project && unzip -oq /content/colab_package.zip && ls

## 3. Greek FLEURS: train and dev audio (about 2 GB)

In [ ]:
%cd /content/project
!python src/prepare_fleurs.py --langs el_gr --splits train dev

## 4. Smoke test (about 2 minutes). Check it trains and exports before the long run.

In [ ]:
!python src/finetune_lora.py --manifest data/manifests/fleurs_el_gr.csv --name smoke --max-train 64 --epochs 1 --lora-targets q_proj,k_proj,v_proj,out_proj,fc1,fc2 --dev-wer-items 0 2>&1 | grep -v -i warn | tail -8

## 5. The real run (3 epochs, all linear layers)
Progress is printed every 10 steps. `models/small-lora-el-all/training_log.csv` gets a row per epoch. Expect a little more than the 25 minutes of the main run.

In [ ]:
!rm -rf models/smoke
!python -u src/finetune_lora.py --manifest data/manifests/fleurs_el_gr.csv --name small-lora-el-all --lora-targets q_proj,k_proj,v_proj,out_proj,fc1,fc2 2>&1 | grep -v -i warn

## 6. Download the result

In [ ]:
!cat models/small-lora-el-all/training_log.csv
!cd models && zip -rq /content/small-lora-el-all.zip small-lora-el-all/ct2 small-lora-el-all/adapter small-lora-el-all/training_log.csv small-lora-el-all/training_config.json
from google.colab import files
files.download("/content/small-lora-el-all.zip")

If the download is slow or fails, copy it to Google Drive instead:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp /content/small-lora-el-all.zip /content/drive/MyDrive/
```